In [1]:
import torch
import chess.pgn

from src.model import ConvNet
from src.predict import get_uci_move
from src.dataclass import preprocess_csv_to_tensors, ChessDataset

In [ ]:
games = load_pgn("data/pgn/lichess_elite_2018-01.pgn")
create_csv_dataset(games, name="test")

In [ ]:
preprocess_csv_to_tensors("data/csv/test.csv", "data/pt/test.pt")

In [2]:
val_dataset = ChessDataset("data/pt/test.pt")
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConvNet(input_channels=12, output_size=4544)
checkpoint = torch.load(f"models/model.99.pth")
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

ConvNet(
  (conv1): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=4544, bias=True)
)

In [4]:
game = chess.pgn.Game()
game.headers["Event"] = "Evaluation Game"
game.headers["Site"] = "Local"
game.headers["White"] = "Human"
game.headers["Black"] = "Model"

node = game

board = chess.Board()
while not board.is_game_over():
    move = get_uci_move(board, model, device)
    board.push(move)
    node = node.add_variation(move)
print(game)
print("\nGame over:", board.result())
with open("data/eval/version_a_1.pgn", "w") as pgn_file:
    print(game, file=pgn_file)

print("Game saved to game_output.pgn — upload it to Lichess to review!")

[Event "Evaluation Game"]
[Site "Local"]
[Date "????.??.??"]
[Round "?"]
[White "Human"]
[Black "Model"]
[Result "*"]

1. d4 d5 2. Nf3 Nf6 3. c4 e6 4. g3 Be7 5. Bg2 O-O 6. O-O dxc4 7. Ne5 Nc6 8. Bxc6 bxc6 9. Nxc6 Qd6 10. Na5 Rb8 11. Nxc4 Qd8 12. Nc3 Ba6 13. b3 Nd5 14. Bd2 Nxc3 15. Bxc3 Bf6 16. Rc1 Bxc4 17. bxc4 Rb2 18. Bxb2 Qb8 19. Rc2 Qxb2 20. Qd3 Qb6 21. e4 Be7 22. d5 Bb4 23. Kg2 a5 24. h4 h6 25. f3 Rd8 26. Rc3 Bxc3 27. Qxc3 Qd4 28. Qxd4 e5 29. Qe3 Rb8 30. g4 Rb2+ 31. Qf2 Rb1 32. Qg3 g6 33. Rf2 Kh7 34. h5 gxh5 35. gxh5 f6 36. Qg6+ Kh8 37. c5 Rf1 38. Rxf1 a4 39. Qxf6+ Kh7 40. Qa6 c6 41. dxc6 Kg7 42. Rb1 a3 43. c7 Kf8 44. Qd3 Ke7 45. Qxa3 Kd7 46. Rb8 Kxc7 47. Re8 Kc6 48. Ra8 Kb7 49. Ra4 Kc6 50. Qb3 Kxc5 51. Rc4+ Kd6 52. Qd3+ Ke6 53. Qa3 Kf7 54. Qb3 Ke6 55. Qb6+ Kf7 56. Qf2 Ke6 57. Qg3 Kd6 58. Qf2 Ke6 59. Rc6+ Kf7 60. a4 Ke7 61. a5 Kd7 62. Rc4 Kd6 63. Ra4 Kc6 64. Rc4+ Kb5 65. Rc8 Kb4 66. Rc7 Ka3 67. Rc5 Kb4 68. Rc8 Kxa5 69. Rd8 Ka6 70. Qg3 Kb7 71. Rd2 Kc6 72. Kh3 Kc5 73. Re2 Kd4 74. Rg2